# Notebook for testing Database Operations

This notebook allows you to connect to a database using the `DatabaseSessionManager` and test database operations, select and queries.

To use this notebook you need to set the `DATABASE_URI` env variable to you database connection string: 

```export DATABASE_URI=postgresql://<user_name>:<password>@<host>:<local_port>/<database_name>```

> NOTE: Jupyter kernels inherit variables from parent server process at STARTUP.  If you created this enviornmental variable _after_ launching Jupyter, you will need to restart the kernel.  If this is too finicky for you (inside VSCode, for example).  Copy the `sample.env` file in this directory `development/genomicsdb-schema` to `.env` and then edit the `DATABASE_URI` value.

The `DATABASE_URI` should then be obtained using our `Settings` object, just so that whole notebook works within our project infrastrcuture.

In [1]:
# get DATABASE URI

from niagads.settings.core import CustomSettings

class Settings(CustomSettings):
    DATABASE_URI: str

In [2]:
# test a connection string 
from niagads.database.session import DatabaseSessionManager

manager = DatabaseSessionManager(connection_string=Settings.from_env().DATABASE_URI)
await manager.test_connection()

True

## Test SQLAlchemy Query leveraging SQLAlchemy models - Reference.ExternalDatabase

In [3]:
from niagads.database.genomicsdb.schema.reference.externaldb import ExternalDatabase
from sqlalchemy import or_, select

db_name = 'kegg'

async with manager.session_ctx() as session:
    stmt = select(ExternalDatabase.name, ExternalDatabase.version).where(
    or_(
        ExternalDatabase.name.ilike(f"%{db_name}%"),
        ExternalDatabase.database_key == db_name.upper(),
    )
)
    result = (await session.execute(stmt)).mappings().all()
    print(result)

[{'name': 'KEGG: Kyoto Encyclopedia of Genes and Genomes', 'version': '117.0'}]


## Test Search Mixin

### Reference.OntologyTerm

In [4]:
import json

from niagads.common.search.models.record import SearchResultRecord
from niagads.database.genomicsdb.schema.reference.ontology import OntologyTerm
from pydantic import TypeAdapter

search_term = "alzheimers disease"
allow_fuzzy = True

async with manager.session_ctx() as session:
    matches: SearchResultRecord = await OntologyTerm.search(session, search_term, allow_fuzzy=allow_fuzzy)
    
    
# Print results.  
# First we need to create a serializer/validator specifically for a list of results

adapter = TypeAdapter(list[SearchResultRecord])
print(json.dumps(adapter.dump_python(matches, mode="json"), indent=4))


[
    {
        "id": "MESH:D000544",
        "record_details": {
            "label": "Alzheimer Disease",
            "description": "A degenerative disease of the BRAIN characterized by the insidious onset of DEMENTIA. Impairment of MEMORY, judgment, attention span, and problem solving skills are followed by severe APRAXIAS and a global loss of cognitive abilities. The condition primarily occurs after age 60, and is marked pathologically by severe cortical atrophy and the triad of SENILE PLAQUES; NEUROFIBRILLARY TANGLES; and NEUROPIL THREADS. (From Adams et al., Principles of Neurology, 6th ed, pp1049-57)",
            "annotation": {
                "iri": "http://id.nlm.nih.gov/mesh/2026/D000544",
                "synonyms": [
                    "Alzheimer Disease",
                    "Alzheimer-Type Dementia (ATD)",
                    "Dementia, Alzheimer-Type (ATD)",
                    "Alzheimer Type Dementia (ATD)",
                    "Alzheimer Syndrome",
               

In [6]:
# include/exclude specific ontologies

async with manager.session_ctx() as session:
    matches: SearchResultRecord = await OntologyTerm.search(session, search_term, allow_fuzzy=allow_fuzzy, include_ontology=["doid"])
    
    
# Print results.  
# First we need to create a serializer/validator specifically for a list of results

adapter = TypeAdapter(list[SearchResultRecord])
print(json.dumps(adapter.dump_python(matches, mode="json"), indent=4))

[
    {
        "id": "DOID:10652",
        "record_details": {
            "label": "Alzheimer's disease",
            "description": "A tauopathy that is characterized by memory lapses, confusion, emotional instability and progressive loss of mental ability and results in progressive memory loss, impaired thinking, disorientation, and changes in personality and mood starting and leads in advanced cases to a profound decline in cognitive and physical functioning and is marked histologically by the degeneration of brain neurons especially in the cerebral cortex and by the presence of neurofibrillary tangles and plaques containing beta-amyloid.",
            "annotation": {
                "iri": "http://purl.obolibrary.org/obo/DOID_10652",
                "synonyms": [
                    "Alzheimer disease",
                    "Alzheimers dementia"
                ]
            }
        },
        "record_type": "ONTOLOGY_TERM",
        "matched_text": "Alzheimer's disease",
       